# Session-Level Population Analysis

This notebook demonstrates session-level analysis of MSN cell populations using the `Session` class.

## Key Features

### 🔧 Architectural Improvements
- **Separated data generation from plotting**: Data methods return dictionaries that can be reused
- **Using exemplar session**: Automatically selects session with most cells (fi211110a with 113 cells)
- **Modular design**: Easy to generate data once and create multiple visualizations

### 📊 Analysis Capabilities

1. **Single Condition Heatmap**: All cells for a specific trial type and direction
2. **Left vs Right Comparison**: Same cells, both directions, maintaining cell order
3. **Trial Type Comparison**: GO, STOP, CONT with appropriate alignments (3×2 grid)
4. **SSD-Separated Analysis**: STOP or CONT trials separated by Stop Signal Delay (4×2 grid per trial type)

All visualizations use:
- **Plasma colormap** for optimal contrast
- **Global normalization**: All firing rates normalized by the maximum across all cells (preserves relative differences)
- **Peak-based sorting** for temporal structure visualization
- **Gaussian smoothing** (sigma=bin_size, truncate=2) for clean PSTHs

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import holoviews as hv
import hvplot.pandas
from pathlib import Path
from holoviews import opts
from bokeh.io import output_notebook
from bokeh.palettes import Plasma256
import sys

# Add parent directory to path to import Cell and PopulationAnalyzer
sys.path.append(str(Path.cwd()))

font_dict = {'title': 16, 'labels': 14, 'ticks': 12, 'legend': 12}
hv.opts.defaults(
    hv.opts.Curve(width=800, height=400, tools=['hover'], fontsize=font_dict),
    hv.opts.Image(width=800, height=600, tools=['hover'], cmap='Plasma', colorbar=True, fontsize=font_dict),
)

output_notebook()
hv.extension('bokeh')

print("Imports loaded successfully!")

Loading BokehJS ...

Imports loaded successfully!


In [3]:
# Load MSN cell database
monkey = 'fiona'  # 'yasmin' or 'fiona'

# save_path = Path.cwd().parent / 'data' / 'unified_cell_trial_data' / 'before_neural_data_fix_bkp'
save_path = Path.cwd().parent / 'data' / 'unified_cell_trial_data' 
pickle_file = save_path / f'msn_{monkey}_cell_trial_data.pkl'

cell_df = pd.read_pickle(pickle_file)

print(f"MSN DataFrame loaded from: {pickle_file}")
print(f"DataFrame shape: {cell_df.shape}")
print(f"\nUnique sessions: {sorted(cell_df['trial_session'].unique())}")
print(f"Number of cells: {cell_df['cell_ID'].nunique()}")
print(f"Number of sessions: {cell_df['trial_session'].nunique()}")

cell_df.head()

MSN DataFrame loaded from: /home/barak/Projects/population_analysis/data/unified_cell_trial_data/msn_fiona_cell_trial_data.pkl
DataFrame shape: (770705, 27)

Unique sessions: ['fi210701a', 'fi210704a', 'fi210705a', 'fi210707a', 'fi210715a', 'fi210718a', 'fi210719a', 'fi210720a', 'fi210721a', 'fi210722a', 'fi210725a', 'fi210726a', 'fi210727a', 'fi210728a', 'fi210729a', 'fi210801a', 'fi210809a', 'fi210810a', 'fi210811a', 'fi210812a', 'fi210815a', 'fi210816a', 'fi210817a', 'fi210818a', 'fi210819a', 'fi210822a', 'fi210823a', 'fi210824a', 'fi210830a', 'fi210831a', 'fi210901a', 'fi210902a', 'fi210907a', 'fi210908a', 'fi210909a', 'fi210927a', 'fi211010a', 'fi211012a', 'fi211014a', 'fi211017a', 'fi211018a', 'fi211019a', 'fi211020a', 'fi211021a', 'fi211025a', 'fi211026a', 'fi211102a', 'fi211103a', 'fi211104a', 'fi211108a', 'fi211109a', 'fi211110a', 'fi211111a', 'fi211115a', 'fi211117a', 'fi211118a', 'fi211122a', 'fi211123a', 'fi211124a', 'fi211125a']
Number of cells: 1414
Number of sessions: 60

,cell_ID,cell_type,maestro_ID,problem,grade,filename,trial_name,reaction_time,go_cue,stop_cue,...,trial_length,screen_rotation,saccades,blinks,dir,neural_data,session,plexon_session,trial_number,trial_session
0,9868,msn,3,NaN,8,fi210824a.0614,CONT_L_SSD2,328.0,1054,1186.0,...,2205,0.0,"[[51, 129], [1382, 1457], [1428, 1465]]",None,180,[],fi210824,a,614,fi210824a
1,9869,msn,4,NaN,8,fi210824a.0614,CONT_L_SSD2,328.0,1054,1186.0,...,2205,0.0,"[[51, 129], [1382, 1457], [1428, 1465]]",None,180,[2197.27],fi210824,a,614,fi210824a
2,9872,msn,7,NaN,8,fi210824a.0614,CONT_L_SSD2,328.0,1054,1186.0,...,2205,0.0,"[[51, 129], [1382, 1457], [1428, 1465]]",None,180,"[925.18, 985.9, 994.25, 2055.17]",fi210824,a,614,fi210824a
3,9873,msn,8,NaN,7,fi210824a.0614,CONT_L_SSD2,328.0,1054,1186.0,...,2205,0.0,"[[51, 129], [1382, 1457], [1428, 1465]]",None,180,"[904.25, 1356.03, 1693.2, 1910.72]",fi210824,a,614,fi210824a
4,9876,msn,11,NaN,8,fi210824a.0614,CONT_L_SSD2,328.0,1054,1186.0,...,2205,0.0,"[[51, 129], [1382, 1457], [1428, 1465]]",None,180,[1076.68],fi210824,a,614,fi210824a


In [4]:
cell_df['grade'].value_counts()

grade
8    584982
7    161227
6     24496
Name: count, dtype: int64

In [5]:
cell_df['neural_data'].apply(lambda x: isinstance(x, list) and len(x) == 0).sum() / cell_df.shape[0]

np.float64(0.11091403325526628)

In [7]:
tmp = cell_df[cell_df['neural_data'].apply(lambda x: isinstance(x, list) and len(x) == 0)]
tmp[tmp['session'] == 'fi211109']

,cell_ID,cell_type,maestro_ID,problem,grade,filename,trial_name,reaction_time,go_cue,stop_cue,...,trial_length,screen_rotation,saccades,blinks,dir,neural_data,session,plexon_session,trial_number,trial_session
581487,2013,pu msn,30,NaN,7,fi211109a.0270,GO_L,157.0,977,NaN,...,2128,0.0,"[[150, 225], [834, 895], [1134, 1211], [1704, ...",None,180,[],fi211109,a,270,fi211109a
581498,2032,pu msn,49,NaN,8,fi211109a.0270,GO_L,157.0,977,NaN,...,2128,0.0,"[[150, 225], [834, 895], [1134, 1211], [1704, ...",None,180,[],fi211109,a,270,fi211109a
581568,2012,pu msn,29,NaN,8,fi211109a.0210,GO_R,213.0,1026,NaN,...,2177,0.0,"[[28, 101], [109, 179], [437, 498], [1239, 131...",None,0,[],fi211109,a,210,fi211109a
581569,2013,pu msn,30,NaN,7,fi211109a.0210,GO_R,213.0,1026,NaN,...,2177,0.0,"[[28, 101], [109, 179], [437, 498], [1239, 131...",None,0,[],fi211109,a,210,fi211109a
581579,2032,pu msn,49,NaN,8,fi211109a.0210,GO_R,213.0,1026,NaN,...,2177,0.0,"[[28, 101], [109, 179], [437, 498], [1239, 131...",None,0,[],fi211109,a,210,fi211109a
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
632600,2074,pu msn,91,NaN,8,fi211109a.0116,GO_L,113.0,1026,NaN,...,2177,0.0,"[[192, 274], [245, 286], [635, 695], [1139, 12...",None,180,[],fi211109,a,116,fi211109a
632603,1992,msn,9,NaN,8,fi211109a.0869,STOP_R_SSD2,592.0,1046,1154.0,...,1854,0.0,"[[258, 323], [1002, 1058], [1638, 1690]]",None,0,[],fi211109,a,869,fi211109a
632623,2046,pu msn,63,NaN,8,fi211109a.0869,STOP_R_SSD2,592.0,1046,1154.0,...,1854,0.0,"[[258, 323], [1002, 1058], [1638, 1690]]",None,0,[],fi211109,a,869,fi211109a
632628,2052,pu msn,69,NaN,8,fi211109a.0869,STOP_R_SSD2,592.0,1046,1154.0,...,1854,0.0,"[[258, 323], [1002, 1058], [1638, 1690]]",None,0,[],fi211109,a,869,fi211109a


In [3]:
# (cell_df.groupby('cell_ID')['neural_data'].apply(
#     lambda x: x.value_counts().index.tolist() == [[]]
# ).sum()) / cell_df['cell_ID'].nunique() 
# cell_df.groupby('cell_ID')['neural_data'].apply(
#     lambda x: x.value_counts() #.index.tolist() == [[]]
# )
cell_df['neural_data'].apply(lambda x: isinstance(x, list) and len(x) == 0).sum() / cell_df.shape[0]

NameError: name 'cell_df' is not defined

In [5]:
cell_df

,cell_ID,cell_type,maestro_ID,problem,grade,filename,trial_name,reaction_time,go_cue,stop_cue,...,trial_length,screen_rotation,saccades,blinks,dir,neural_data,session,plexon_session,trial_number,trial_session
0,9868,msn,3,NaN,8,fi210824a.0614,CONT_L_SSD2,328.0,1054,1186.0,...,2205,0.0,"[[51, 129], [1382, 1457], [1428, 1465]]",None,180,[],fi210824,a,614,fi210824a
1,9869,msn,4,NaN,8,fi210824a.0614,CONT_L_SSD2,328.0,1054,1186.0,...,2205,0.0,"[[51, 129], [1382, 1457], [1428, 1465]]",None,180,[2197.27],fi210824,a,614,fi210824a
2,9872,msn,7,NaN,8,fi210824a.0614,CONT_L_SSD2,328.0,1054,1186.0,...,2205,0.0,"[[51, 129], [1382, 1457], [1428, 1465]]",None,180,"[925.18, 985.9, 994.25, 2055.17]",fi210824,a,614,fi210824a
3,9873,msn,8,NaN,7,fi210824a.0614,CONT_L_SSD2,328.0,1054,1186.0,...,2205,0.0,"[[51, 129], [1382, 1457], [1428, 1465]]",None,180,"[904.25, 1356.03, 1693.2, 1910.72]",fi210824,a,614,fi210824a
4,9876,msn,11,NaN,8,fi210824a.0614,CONT_L_SSD2,328.0,1054,1186.0,...,2205,0.0,"[[51, 129], [1382, 1457], [1428, 1465]]",None,180,[1076.68],fi210824,a,614,fi210824a
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
984755,2930,pu msn,99,NaN,8,fi211020a.0451,CONT_L_SSD1,199.0,1008,1056.0,...,2159,0.0,"[[1207, 1281]]",None,180,"[78.76, 246.83, 746.31, 892.11, 941.41, 1118.6...",fi211020,a,451,fi211020a
984756,2931,pu msn,100,NaN,8,fi211020a.0451,CONT_L_SSD1,199.0,1008,1056.0,...,2159,0.0,"[[1207, 1281]]",None,180,"[199.58, 234.53, 289.33, 297.48, 591.91, 648.4...",fi211020,a,451,fi211020a
984757,2933,pu msn,102,NaN,8,fi211020a.0451,CONT_L_SSD1,199.0,1008,1056.0,...,2159,0.0,"[[1207, 1281]]",None,180,"[84.61, 316.63, 366.98, 522.18, 640.41, 656.28...",fi211020,a,451,fi211020a
984758,2934,pu msn,103,NaN,8,fi211020a.0451,CONT_L_SSD1,199.0,1008,1056.0,...,2159,0.0,"[[1207, 1281]]",None,180,"[66.76, 141.41, 165.56, 270.36, 482.13, 503.13...",fi211020,a,451,fi211020a


In [6]:
# Add parent directory to path to import Cell and PopulationAnalyzer
import sys
sys.path.insert(0, str(Path.cwd().parent))
import importlib
import session_class
importlib.reload(session_class)
from session_class import Session
from cell_analysis import Cell, PopulationAnalyzer

print("Cell, PopulationAnalyzer, and Session classes imported successfully!")

Cell, PopulationAnalyzer, and Session classes imported successfully!


In [7]:
# Select a session to analyze - use session with most cells
session_cell_counts = cell_df.groupby('trial_session')['cell_ID'].nunique().sort_values(ascending=False)
best_session_id = session_cell_counts.index[0]

print(f"Using session with most cells: {best_session_id}")
print(f"Number of cells in this session: {session_cell_counts.iloc[0]}")

best_session_id = 'fi211109a'
# Get all data for this session
session_data = cell_df[cell_df['trial_session'] == best_session_id]

# Create Session object
session = Session(session_data, verbose=True)

Using session with most cells: fi211110a
Number of cells in this session: 85
Session fi211109a initialized:
  - Number of cells: 59
  - Total trials: 920
  - Trial types: ['CONT', 'GO', 'STOP']
  - Directions: [np.int64(0), np.int64(180)]


## Example 1: Single Condition Heatmap
Plot all cells in the session for STOP trials, right direction (0°), with cells sorted by peak activity time.

In [8]:
# Plot heatmap for STOP trials, right direction
heatmap = session.plot_population_heatmap(
    epok=[-200, 700],
    bin_size=10,
    alignment_point='stop_cue',
    trial_type='STOP',
    direction=0,
    success_only=True,
    normalize=True,
    sort_by_peak=True
)

heatmap

:HeatMap   [columns,index]   (value)

## Example 1b: Spike Counts Heatmap
Similar to the PSTH heatmap above, but showing raw spike counts per bin instead of smoothed firing rates. This is useful for examining the discrete spike patterns.

In [9]:
# Plot spike counts heatmap for STOP trials, right direction
# Using smaller bin size (1ms) to see fine-grained spike patterns
spike_counts_heatmap = session.plot_population_spike_counts_heatmap(
    epok=[-200, 700],
    bin_size=1,
    alignment_point='stop_cue',
    trial_type='STOP',
    direction=0,
    success_only=True,
    normalize=True,
    sort_by_peak=True
)

spike_counts_heatmap.opts(height=300)

:HeatMap   [columns,index]   (value)

In [10]:
# Generate spike counts data for STOP trials, right direction
spike_counts_data = session.get_population_spike_counts_data(
    epok=[-1000, 1000],
    bin_size=1,
    alignment_point='stop_cue',
    trial_type='STOP',
    direction=0,
    success_only=True,
    normalize=True,
    sort_by_peak=True
)

len(spike_counts_data['cell_ids']), spike_counts_data['sort_idx'].shape
spike_counts_data
session.plot_population_spike_counts_heatmap(spike_counts_data).opts(height=300)


:HeatMap   [columns,index]   (value)

In [11]:
session.data[
    (session.data['cell_ID'] == 2055)  
    # (session.data['type'] == 'STOP') & 
    # (session.data['dir'] == 0) &
    # (session.data['trial_failed'] == False)
]['neural_data'].value_counts().index.to_list() == [[]]

False

In [12]:
session.get_cells_with_no_spikes(as_percentage=True)

np.float64(0.0)

## Example 2: Left vs Right Comparison
Compare left and right directions with the same cell ordering (based on left direction peaks).

In [13]:
# Compare left vs right for STOP trials
left_right_comparison = session.plot_left_right_comparison(
    epok=[-200, 700],
    bin_size=10,
    alignment_point='stop_cue',
    trial_type='STOP',
    success_only=True,
    smooth=True,
    normalize=True
)

left_right_comparison

:Layout
   .HeatMap.I  :HeatMap   [columns,index]   (value)
   .HeatMap.II :HeatMap   [columns,index]   (value)

## Example 3: Trial Type Comparison (3×2 Grid)
Compare all trial types across both directions:
- Rows: GO, STOP, CONT trial types
- Columns: Left (180°) and Right (0°) directions
- GO: aligned to go_cue
- STOP: aligned to stop_cue  
- CONT: aligned to stop_cue

All six heatmaps use the same cell ordering (based on GO left trial peaks).

In [14]:
# Compare GO, STOP, and CONT trials across left and right directions
trial_type_comparison = session.plot_trial_type_comparison(
    epok_go=[-200, 1000],
    epok_stop=[-200, 1000],
    bin_size=10,
    success_only=True,
    smooth=True,
    normalize=True
)

trial_type_comparison

:Layout
   .HeatMap.I   :HeatMap   [columns,index]   (value)
   .HeatMap.II  :HeatMap   [columns,index]   (value)
   .HeatMap.III :HeatMap   [columns,index]   (value)
   .HeatMap.IV  :HeatMap   [columns,index]   (value)
   .HeatMap.V   :HeatMap   [columns,index]   (value)
   .HeatMap.VI  :HeatMap   [columns,index]   (value)

## Example 4a: STOP Trials by SSD (4×2 Grid)
Separate STOP trials by SSD number (Stop Signal Delay):
- Rows: SSD numbers (SSD1, SSD2, SSD3, SSD4)
- Columns: Left (180°) and Right (0°) directions
- All aligned to stop_cue
- All use the same cell ordering (based on GO left trial peaks)

In [15]:
# Plot STOP trials separated by SSD number
stop_ssd_comparison = session.plot_trial_type_by_ssd(
    trial_type='STOP',
    epok_stop=[-200, 700],
    bin_size=10,
    success_only=True,
    smooth=True,
    normalize=True
)

stop_ssd_comparison

:Layout
   .HeatMap.I   :HeatMap   [columns,index]   (value)
   .HeatMap.II  :HeatMap   [columns,index]   (value)
   .HeatMap.III :HeatMap   [columns,index]   (value)
   .HeatMap.IV  :HeatMap   [columns,index]   (value)
   .HeatMap.V   :HeatMap   [columns,index]   (value)
   .HeatMap.VI  :HeatMap   [columns,index]   (value)
   .HeatMap.VII :HeatMap   [columns,index]   (value)
   .Text.I      :Text   [x,y]

## Example 4b: CONT Trials by SSD (4×2 Grid)
Separate CONT trials by SSD number (Stop Signal Delay):
- Rows: SSD numbers (SSD1, SSD2, SSD3, SSD4)
- Columns: Left (180°) and Right (0°) directions
- All aligned to stop_cue
- All use the same cell ordering (based on GO left trial peaks)

In [16]:
# Plot CONT trials separated by SSD number
cont_ssd_comparison = session.plot_trial_type_by_ssd(
    trial_type='CONT',
    epok_stop=[-200, 700],
    bin_size=10,
    success_only=True,
    smooth=True,
    normalize=True
)

cont_ssd_comparison

:Layout
   .HeatMap.I    :HeatMap   [columns,index]   (value)
   .HeatMap.II   :HeatMap   [columns,index]   (value)
   .HeatMap.III  :HeatMap   [columns,index]   (value)
   .HeatMap.IV   :HeatMap   [columns,index]   (value)
   .HeatMap.V    :HeatMap   [columns,index]   (value)
   .HeatMap.VI   :HeatMap   [columns,index]   (value)
   .HeatMap.VII  :HeatMap   [columns,index]   (value)
   .HeatMap.VIII :HeatMap   [columns,index]   (value)